In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install alpaca-trade-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.7/757.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 8.3 MB/s eta 0:00:00
  Created wheel for msgpack: filename=msgpack-1.0.3-cp311-cp311-linux_x86_64.whl size=15688 sha256=29e47f4f4160a932e0401e0a4bc2d2a5ba04cb7ab572118813072bc1a894d05f
  Stored in directory: /root/.cache/pip/wheels/f6/35/da/ed9b26b510235e00e3a3c3bab7bad97b59214729662255ab3d
Successfully built msgpack
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.1.0
    Uninstalling msgpack-1.1.0:
      Successfully uninstalled msgpack-1.1.0
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling

In [3]:
import alpaca_trade_api as tradeapi
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
import numpy as np

In [ ]:
API_KEY = "YOUR_API_KEY_HERE"
SECRET_KEY = "YOUR_SECRET_KEY_HERE"
BASE_URL = "https://api.alpaca.markets"

In [5]:
# Inicializa a API Alpaca com as credenciais lidas do arquivo
api = tradeapi.REST(API_KEY, SECRET_KEY, BASE_URL, api_version='v2')

In [6]:
symbol = "AAPL"
timeframe = "5Min"
data_source = "sip"

In [7]:
clock = api.get_clock()

In [8]:
clock

Clock({   'is_open': False,
    'next_close': '2025-04-22T16:00:00-04:00',
    'next_open': '2025-04-22T09:30:00-04:00',
    'timestamp': '2025-04-22T09:10:13.8550406-04:00'})

In [40]:
# Example: UTC timestamps with 'Z'
now   = datetime.now(timezone.utc)
start = (now - timedelta(hours=3) - timedelta(minutes=15)).isoformat()  # ⇒ '2025-04-21T12:49:12.774699+00:00'
end   = (now - timedelta(minutes=15)).isoformat()  # ⇒ '2025-04-21T13:34:12.774699+00:00'

print("Now (UTC):", now)
print("Start (UTC):", start)
print("End   (UTC):", end)

Now (UTC): 2025-04-22 13:53:57.361563+00:00
Start (UTC): 2025-04-22T10:38:57.361563+00:00
End   (UTC): 2025-04-22T13:38:57.361563+00:00


In [41]:
all_data = []

# Fetch the historical data
bars = api.get_bars(
    symbol,
    timeframe,
    start,
    end,
    feed=data_source
).df

all_data.append(bars)

In [11]:
all_data = pd.concat(all_data)

In [12]:
all_data

,close,high,low,trade_count,open,volume,vwap
timestamp,,,,,,,
2025-04-22 10:00:00+00:00,195.2700,195.2700,195.0100,52,195.2500,1775,195.196081
2025-04-22 10:05:00+00:00,195.1900,195.1900,195.1900,20,195.1900,718,195.190000
2025-04-22 10:10:00+00:00,195.1200,195.1200,195.1200,18,195.1200,715,195.120000
2025-04-22 10:15:00+00:00,195.0500,195.0700,195.0500,22,195.0700,1129,195.057752
2025-04-22 10:20:00+00:00,195.2000,195.2000,195.1200,13,195.1200,453,195.160000
2025-04-22 10:25:00+00:00,194.9500,195.1700,194.9400,27,195.1700,808,195.048662
2025-04-22 10:30:00+00:00,194.9000,195.0000,194.9000,39,194.9500,1230,194.964413
2025-04-22 10:35:00+00:00,194.9800,194.9800,194.9200,46,194.9200,1180,194.940000
2025-04-22 10:40:00+00:00,195.0700,195.2000,195.0700,42,195.1800,1125,195.126000


In [13]:
data = all_data[['vwap', 'trade_count']]

In [14]:
data

,vwap,trade_count
timestamp,,
2025-04-22 10:00:00+00:00,195.196081,52
2025-04-22 10:05:00+00:00,195.190000,20
2025-04-22 10:10:00+00:00,195.120000,18
2025-04-22 10:15:00+00:00,195.057752,22
2025-04-22 10:20:00+00:00,195.160000,13
2025-04-22 10:25:00+00:00,195.048662,27
2025-04-22 10:30:00+00:00,194.964413,39
2025-04-22 10:35:00+00:00,194.940000,46
2025-04-22 10:40:00+00:00,195.126000,42


In [15]:
scaler = joblib.load("/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/minmax_ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min.pkl")

In [16]:
X_VWAP = all_data[['vwap']]

X_VWAP_scaled = scaler.transform(X_VWAP)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [17]:
X_VWAP_scaled

array([[0.24861594],
       [0.24860165],
       [0.24843717],
       [0.2482909 ],
       [0.24853116],
       [0.24826954],
       [0.24807157],
       [0.24801421],
       [0.24845126],
       [0.24826013],
       [0.24838498],
       [0.24828161],
       [0.24760662],
       [0.24760643],
       [0.24723879],
       [0.24721982],
       [0.24747853],
       [0.2478822 ],
       [0.24810624],
       [0.24741831],
       [0.24743585],
       [0.24812162],
       [0.2473842 ],
       [0.24697192],
       [0.24702934],
       [0.24696976],
       [0.24711542],
       [0.24710656],
       [0.24724916],
       [0.24734062],
       [0.24740238],
       [0.24714318],
       [0.24718667],
       [0.24766444],
       [0.24803185],
       [0.24850844]])

In [18]:
X_Trade_Count = all_data[['trade_count']].to_numpy()

In [19]:
X_Trade_Count

array([[  52],
       [  20],
       [  18],
       [  22],
       [  13],
       [  27],
       [  39],
       [  46],
       [  42],
       [  38],
       [  58],
       [  43],
       [ 311],
       [  73],
       [  24],
       [  37],
       [  29],
       [  20],
       [  56],
       [  52],
       [  35],
       [ 124],
       [ 183],
       [  12],
       [1653],
       [ 135],
       [ 311],
       [ 170],
       [ 117],
       [ 129],
       [ 181],
       [  91],
       [ 245],
       [ 314],
       [ 307],
       [ 265]])

In [20]:
X_combined = np.concatenate([X_VWAP_scaled, X_Trade_Count], axis=1)

In [32]:
X_combined

array([[2.40004298e-01, 4.18500000e+03],
       [2.39800449e-01, 4.56300000e+03],
       [2.38939032e-01, 5.47600000e+03],
       [2.38550689e-01, 4.41500000e+03],
       [2.38249657e-01, 4.51800000e+03],
       [2.37716330e-01, 6.39900000e+03],
       [2.37214863e-01, 5.44700000e+03],
       [2.37155290e-01, 6.31600000e+03],
       [2.37072219e-01, 4.20000000e+03],
       [2.38070930e-01, 4.73900000e+03],
       [2.37752730e-01, 3.88000000e+03],
       [2.38223927e-01, 4.71100000e+03],
       [2.38175677e-01, 5.40500000e+03],
       [2.37856554e-01, 5.22900000e+03],
       [2.37504985e-01, 4.59900000e+03],
       [2.37572715e-01, 4.02600000e+03],
       [2.37504591e-01, 3.72900000e+03],
       [2.37417871e-01, 3.98000000e+03],
       [2.37286547e-01, 4.06200000e+03],
       [2.37471191e-01, 3.42500000e+03],
       [2.38139063e-01, 4.78200000e+03],
       [2.38109358e-01, 4.06500000e+03],
       [2.38090844e-01, 3.32800000e+03],
       [2.38674538e-01, 4.75300000e+03],
       [2.386367

In [22]:
X_Tensor = np.expand_dims(X_combined, axis=0)

In [23]:
X_Tensor

array([[[2.48615938e-01, 5.20000000e+01],
        [2.48601649e-01, 2.00000000e+01],
        [2.48437166e-01, 1.80000000e+01],
        [2.48290898e-01, 2.20000000e+01],
        [2.48531156e-01, 1.30000000e+01],
        [2.48269539e-01, 2.70000000e+01],
        [2.48071574e-01, 3.90000000e+01],
        [2.48014209e-01, 4.60000000e+01],
        [2.48451264e-01, 4.20000000e+01],
        [2.48260130e-01, 3.80000000e+01],
        [2.48384980e-01, 5.80000000e+01],
        [2.48281614e-01, 4.30000000e+01],
        [2.47606617e-01, 3.11000000e+02],
        [2.47606434e-01, 7.30000000e+01],
        [2.47238788e-01, 2.40000000e+01],
        [2.47219821e-01, 3.70000000e+01],
        [2.47478532e-01, 2.90000000e+01],
        [2.47882199e-01, 2.00000000e+01],
        [2.48106240e-01, 5.60000000e+01],
        [2.47418310e-01, 5.20000000e+01],
        [2.47435849e-01, 3.50000000e+01],
        [2.48121616e-01, 1.24000000e+02],
        [2.47384203e-01, 1.83000000e+02],
        [2.46971924e-01, 1.2000000

In [31]:
X_Tensor = tf.convert_to_tensor(X_Tensor)

In [32]:
X_Tensor

<tf.Tensor: shape=(1, 36, 2), dtype=float64, numpy=
array([[[2.48615938e-01, 5.20000000e+01],
        [2.48601649e-01, 2.00000000e+01],
        [2.48437166e-01, 1.80000000e+01],
        [2.48290898e-01, 2.20000000e+01],
        [2.48531156e-01, 1.30000000e+01],
        [2.48269539e-01, 2.70000000e+01],
        [2.48071574e-01, 3.90000000e+01],
        [2.48014209e-01, 4.60000000e+01],
        [2.48451264e-01, 4.20000000e+01],
        [2.48260130e-01, 3.80000000e+01],
        [2.48384980e-01, 5.80000000e+01],
        [2.48281614e-01, 4.30000000e+01],
        [2.47606617e-01, 3.11000000e+02],
        [2.47606434e-01, 7.30000000e+01],
        [2.47238788e-01, 2.40000000e+01],
        [2.47219821e-01, 3.70000000e+01],
        [2.47478532e-01, 2.90000000e+01],
        [2.47882199e-01, 2.00000000e+01],
        [2.48106240e-01, 5.60000000e+01],
        [2.47418310e-01, 5.20000000e+01],
        [2.47435849e-01, 3.50000000e+01],
        [2.48121616e-01, 1.24000000e+02],
        [2.47384203e-01,

In [33]:
model = load_model("/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min+fm=vwap+sm=trade_count+tm=+r=36+sort=False+rfm=False+rsm=False+rtm=False+d=+st=minmax+cts=[0]+Lb=True+e=500+es=True+cb=val_accuracy+p=100+bs=128+tl=0.41606152057647705+ta=0.8365758657455444.keras")

In [34]:
predictions = model.predict(X_Tensor)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step


In [35]:
predictions

array([[0.95927376]], dtype=float32)

In [36]:
decisive_sensibility = 0.5

predicted_classes = (predictions >= decisive_sensibility).astype(int)

In [37]:
print(predictions)
print(predicted_classes)

[[0.95927376]]
[[1]]
